In [ ]:
!pip install -q pandas openpyxl sentence-transformers transformers

In [ ]:
import pandas as pd
import numpy as np
import re
import torch
from sentence_transformers import SentenceTransformer
from IPython.display import display

import warnings
warnings.filterwarnings('ignore') # Hides pandas slicing warnings

In [ ]:
class DataIngestionAgent:
    def __init__(self, univ_path, scholarship_path):
        self.univ_path = univ_path
        self.scholarship_path = scholarship_path

    def load_universities(self, domain_name):
        print(f"[Ingestion] Accessing sheet: {domain_name}...")

        # FIX: The real data has headers on the 4th row (index 3)
        df = pd.read_excel(self.univ_path, sheet_name=domain_name, header=3)
        df.columns = df.columns.astype(str).str.strip()

        # Rename INSTITUTION to match scholarship sheets
        if 'INSTITUTION' in df.columns:
            df.rename(columns={'INSTITUTION': 'University / Partners'}, inplace=True)

        return df

    def load_scholarships(self):
        # Master scholarship list has standard formatting (header=0)
        df = pd.read_excel(self.scholarship_path)
        df.columns = df.columns.astype(str).str.strip()
        return df

In [ ]:
class ProfilingAgent:
    def run(self, user_data):
        return {
            "domain": str(user_data.get("domain", "")),
            "gpa": float(user_data.get("gpa", 0.0)),
            "ielts": float(user_data.get("ielts", 0.0)),
            "degree": str(user_data.get("degree_level", "Masters")),
            "gre": str(user_data.get("gre", "No")),
            "interest": str(user_data.get("specific_interest", ""))
        }

In [ ]:
class UniversityFilterAgent:
    def _extract_rank(self, rank_val):
        """Extracts the first number from strings like '51-100' or '1'."""
        try:
            match = re.search(r'\d+', str(rank_val))
            return int(match.group()) if match else 9999
        except:
            return 9999

    def run(self, univ_df):
        df = univ_df.copy()
        rank_col = '2026' if '2026' in df.columns else 'Rank'

        # FIX: Create a purely numeric column for accurate sorting
        df['Numeric_Rank'] = df[rank_col].apply(self._extract_rank)

        # Sort by best rank and take top 20
        top_20 = df.sort_values(by='Numeric_Rank', ascending=True).head(20)
        return top_20

In [ ]:
class ScholarshipFilterAgent:
    def _extract_score(self, score_str):
        """Extracts '6.5' from strings like '6.5 / 79 iBT'."""
        try:
            match = re.search(r'\d+(\.\d+)?', str(score_str))
            return float(match.group()) if match else 0.0
        except:
            return 0.0

    def run(self, scholarship_df, profile):
        df = scholarship_df.copy()

        # 1. Degree Level
        df = df[df['Degree Level'].str.contains(profile['degree'], case=False, na=False)]

        # 2. IELTS / TOEFL (Extracting numbers safely)
        df['Parsed_IELTS'] = df['Min IELTS / TOEFL'].apply(self._extract_score)
        df = df[df['Parsed_IELTS'] <= profile['ielts']]

        # 3. GRE Requirement
        if profile['gre'].lower() == "no":
            df = df[~df['GRE Required?'].str.contains("Required", case=False, na=False)]

        # 4. Field Restrictions (Looking for "Open" or specific domain overlap)
        def field_match(x):
            x_str = str(x).lower()
            if 'open' in x_str or 'none' in x_str:
                return True
            # Check if main domain words exist in restriction
            domain_words = profile['domain'].lower().replace('&', '').split()
            return any(len(w) > 3 and w in x_str for w in domain_words)

        df = df[df['Field Restrictions'].apply(field_match)]

        return df

In [ ]:
class MatchingAgent:
    def __init__(self):
        # Loads the embedding model only once
        print("[System] Loading Embedding Model...")
        self.embed_model = SentenceTransformer("all-MiniLM-L6-v2")

    def run(self, filtered_univs, filtered_scholarships, profile):
        # Step 1: Attempt to find exact University matches
        matched_univs = filtered_univs['University / Partners'].tolist()

        def is_partner(sch_univ):
            # Accounts for "Most US Universities" or exact names
            sch_str = str(sch_univ).lower()
            if 'all' in sch_str or 'most' in sch_str:
                return True
            return any(str(u).lower() in sch_str for u in matched_univs)

        mask = filtered_scholarships['University / Partners'].apply(is_partner)
        best_matches = filtered_scholarships[mask].copy()

        # Step 2: Fallback if too few specific matches
        if len(best_matches) < 5:
            best_matches = filtered_scholarships.copy()

        # Step 3: Semantic Match on Description vs User Interest
        if len(best_matches) > 0 and profile['interest']:
            descriptions = best_matches['Description'].fillna("").tolist()
            desc_vecs = self.embed_model.encode(descriptions)
            user_vec = self.embed_model.encode([profile['interest']])

            # Dot product for similarity
            scores = np.dot(desc_vecs, user_vec.T).flatten()
            best_matches['semantic_score'] = scores
            best_matches = best_matches.sort_values(by='semantic_score', ascending=False)

        return best_matches.head(5)

In [ ]:
class MatchingAgent:
    def __init__(self):
        # Loads the embedding model only once
        print("[System] Loading Embedding Model...")
        self.embed_model = SentenceTransformer("all-MiniLM-L6-v2")

    def run(self, filtered_univs, filtered_scholarships, profile):
        # Step 1: Attempt to find exact University matches
        matched_univs = filtered_univs['University / Partners'].tolist()

        def is_partner(sch_univ):
            # Accounts for "Most US Universities" or exact names
            sch_str = str(sch_univ).lower()
            if 'all' in sch_str or 'most' in sch_str:
                return True
            return any(str(u).lower() in sch_str for u in matched_univs)

        mask = filtered_scholarships['University / Partners'].apply(is_partner)
        best_matches = filtered_scholarships[mask].copy()

        # Step 2: Fallback if too few specific matches
        if len(best_matches) < 5:
            best_matches = filtered_scholarships.copy()

        # Step 3: Semantic Match on Description vs User Interest
        if len(best_matches) > 0 and profile['interest']:
            descriptions = best_matches['Description'].fillna("").tolist()
            desc_vecs = self.embed_model.encode(descriptions)
            user_vec = self.embed_model.encode([profile['interest']])

            # Dot product for similarity
            scores = np.dot(desc_vecs, user_vec.T).flatten()
            best_matches['semantic_score'] = scores
            best_matches = best_matches.sort_values(by='semantic_score', ascending=False)

        return best_matches.head(5)

In [ ]:
class EgyptianScholarshipSystem:
    def __init__(self, univ_file, scholarship_file):
        self.ingestor = DataIngestionAgent(univ_file, scholarship_file)
        self.profiler = ProfilingAgent()
        self.univ_filter = UniversityFilterAgent()
        self.sch_filter = ScholarshipFilterAgent()
        self.matcher = MatchingAgent()

        self.scholarship_data = self.ingestor.load_scholarships()

    def process_request(self, user_input):
        profile = self.profiler.run(user_input)

        # Load exactly the sheet requested by the user interface
        univ_data = self.ingestor.load_universities(profile['domain'])

        top_univs = self.univ_filter.run(univ_data)
        possible_scholarships = self.sch_filter.run(self.scholarship_data, profile)

        final_results = self.matcher.run(top_univs, possible_scholarships, profile)
        return final_results

In [ ]:
# Initialize with your exact file names
system = EgyptianScholarshipSystem("/content/Universities.xlsx", "/content/Scholarships.xlsx")

# Interface Input Simulation
user_query = {
    "domain": "Engineering - Electrical & Elec",
    "gpa": 3.9,
    "ielts": 7.0,
    "degree_level": "Masters",
    "gre": "No",
    "specific_interest": "Brain-Computer Interfaces and Robotics"
}

# Run Pipeline
top_5 = system.process_request(user_query)

# Display final formatted output
print("\n=== TOP 5 PERSONALIZED SCHOLARSHIPS ===")
display(top_5[[
    'Scholarship Name',
    'University / Partners',
    'Funding Type',
    'Deadline Month'
]])

[System] Loading Embedding Model...


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


[Ingestion] Accessing sheet: Engineering - Electrical & Elec...

=== TOP 5 PERSONALIZED SCHOLARSHIPS ===


,Scholarship Name,University / Partners,Funding Type,Deadline Month
16,Sawiris Graduate Scholarship,"Top-50 Global Unis in US, UK, Germany, Switzer...",Full,March
29,Khalifa University Scholarship (UAE),"Khalifa University of Science and Technology, ...",Full,February
13,KAUST Fellowship (Saudi Arabia),KAUST – King Abdullah Univ. of Science & Techn...,Full,January
20,Aga Khan Foundation (AKF) ISP,Any Reputable University (worldwide),Partial (50% grant + 50% loan),March
46,AAUW International Fellowship (USA – Women only),Accredited US Universities,Partial to Full,November
